# PyBullet Tabletop Stacking: Multi-View Video Capture

This notebook uses `phys_sim`, SI units, PyBullet rigid-body dynamics, and TinyRenderer to simulate a block stack settling on a tabletop. It models gravity, contact, friction, restitution, and solver iterations, then records front, side, back, and top videos.

In [ ]:
import sys
from pathlib import Path

assert "phys_sim" in sys.executable, (
    f"Expected the phys_sim virtual environment, but got: {sys.executable}\n"
    "In Jupyter, select the kernel named 'phys_sim'."
)

import numpy as np
import pybullet as p
import pybullet_data
import imageio.v2 as imageio
from IPython.display import Video, display

print(f"Python executable: {sys.executable}")
print(f"PyBullet data path: {pybullet_data.getDataPath()}")


## Scenario Parameters

Blocks are released sequentially above the tabletop. The support contacts, small offsets, and friction determine whether the stack settles or topples.

In [ ]:
OUTPUT_DIR = Path("pybullet_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

GRAVITY = -9.80665
SIM_HZ = 360
TIME_STEP = 1.0 / SIM_HZ
VIDEO_FPS = 60
STEPS_PER_FRAME = SIM_HZ // VIDEO_FPS
DURATION_SEC = 5.0
N_STEPS = int(DURATION_SEC * SIM_HZ)

TABLE_LENGTH = 1.20
TABLE_WIDTH = 0.80
TABLE_THICKNESS = 0.06
TABLE_TOP_Z = 0.75
TABLE_CENTER_Z = TABLE_TOP_Z - TABLE_THICKNESS / 2.0
TABLE_RESTITUTION = 0.08
TABLE_FRICTION = 0.82

BLOCK_SIZE = [0.070, 0.070, 0.045]
BLOCK_MASS = 0.060
BLOCK_FRICTION = 0.85
BLOCK_RESTITUTION = 0.05
RELEASE_INTERVAL_SEC = 0.70

IMG_WIDTH = 640
IMG_HEIGHT = 368

print(f"Writing outputs to: {OUTPUT_DIR.resolve()}")


In [ ]:
def connect_pybullet(deformable=False):
    if p.isConnected():
        p.disconnect()
    client = p.connect(p.DIRECT)
    if deformable:
        p.resetSimulation(p.RESET_USE_DEFORMABLE_WORLD, physicsClientId=client)
    else:
        p.resetSimulation(physicsClientId=client)
    p.setAdditionalSearchPath(pybullet_data.getDataPath(), physicsClientId=client)
    p.setGravity(0, 0, GRAVITY, physicsClientId=client)
    p.setTimeStep(TIME_STEP, physicsClientId=client)
    p.setPhysicsEngineParameter(
        fixedTimeStep=TIME_STEP,
        numSolverIterations=180,
        numSubSteps=2,
        deterministicOverlappingPairs=1,
        physicsClientId=client,
    )
    return client


def create_table(client):
    p.loadURDF("plane.urdf", physicsClientId=client)
    collision = p.createCollisionShape(
        p.GEOM_BOX,
        halfExtents=[TABLE_LENGTH / 2, TABLE_WIDTH / 2, TABLE_THICKNESS / 2],
        physicsClientId=client,
    )
    visual = p.createVisualShape(
        p.GEOM_BOX,
        halfExtents=[TABLE_LENGTH / 2, TABLE_WIDTH / 2, TABLE_THICKNESS / 2],
        rgbaColor=[0.58, 0.40, 0.24, 1.0],
        physicsClientId=client,
    )
    table_id = p.createMultiBody(
        baseMass=0,
        baseCollisionShapeIndex=collision,
        baseVisualShapeIndex=visual,
        basePosition=[0, 0, TABLE_CENTER_Z],
        physicsClientId=client,
    )
    p.changeDynamics(
        table_id,
        -1,
        restitution=TABLE_RESTITUTION,
        lateralFriction=TABLE_FRICTION,
        spinningFriction=0.01,
        rollingFriction=0.01,
        physicsClientId=client,
    )
    return table_id

def camera_matrices(view_name, target):
    cameras = {
        "front": {"eye": [0.0, -1.65, 1.08], "up": [0, 0, 1], "fov": 50},
        "side": {"eye": [1.65, 0.0, 1.08], "up": [0, 0, 1], "fov": 50},
        "back": {"eye": [0.0, 1.65, 1.08], "up": [0, 0, 1], "fov": 50},
        "top": {"eye": [0.0, 0.0, 2.35], "up": [0, 1, 0], "fov": 44},
    }
    spec = cameras[view_name]
    view = p.computeViewMatrix(spec["eye"], target, spec["up"])
    proj = p.computeProjectionMatrixFOV(
        fov=spec["fov"],
        aspect=IMG_WIDTH / IMG_HEIGHT,
        nearVal=0.02,
        farVal=5.0,
    )
    return view, proj


CAMERA_NAMES = ["front", "side", "back", "top"]


def render_rgb(client, view_name, target):
    view, proj = camera_matrices(view_name, target)
    _, _, rgba, _, _ = p.getCameraImage(
        width=IMG_WIDTH,
        height=IMG_HEIGHT,
        viewMatrix=view,
        projectionMatrix=proj,
        renderer=p.ER_TINY_RENDERER,
        lightDirection=[-0.5, -0.4, -1.0],
        physicsClientId=client,
    )
    rgba = np.asarray(rgba, dtype=np.uint8).reshape(IMG_HEIGHT, IMG_WIDTH, 4)
    return rgba[:, :, :3]


def make_writers(prefix):
    paths = {name: OUTPUT_DIR / f"{prefix}_{name}.mp4" for name in CAMERA_NAMES}
    writers = {
        name: imageio.get_writer(path, fps=VIDEO_FPS, codec="libx264", quality=8, macro_block_size=16)
        for name, path in paths.items()
    }
    return paths, writers

def block_pose(index):
    offsets = [
        [0.000, 0.000],
        [0.010, -0.004],
        [-0.006, 0.008],
        [0.012, 0.006],
        [-0.010, -0.008],
        [0.004, 0.012],
    ]
    x, y = offsets[index]
    z = TABLE_TOP_Z + BLOCK_SIZE[2] / 2 + index * BLOCK_SIZE[2]
    yaw = [0.0, 0.08, -0.06, 0.12, -0.10, 0.05][index]
    return [x, y, z], p.getQuaternionFromEuler([0, 0, yaw])


def create_block(client, index):
    pos, orn = block_pose(index)
    pos = [pos[0], pos[1], TABLE_TOP_Z + 0.34 + index * 0.015]
    half = [v / 2 for v in BLOCK_SIZE]
    colors = [
        [0.88, 0.15, 0.12, 1], [0.10, 0.55, 0.85, 1], [0.12, 0.70, 0.25, 1],
        [0.95, 0.70, 0.10, 1], [0.48, 0.22, 0.75, 1], [0.90, 0.38, 0.16, 1],
    ]
    collision = p.createCollisionShape(p.GEOM_BOX, halfExtents=half, physicsClientId=client)
    visual = p.createVisualShape(p.GEOM_BOX, halfExtents=half, rgbaColor=colors[index], physicsClientId=client)
    body = p.createMultiBody(BLOCK_MASS, collision, visual, pos, orn, physicsClientId=client)
    p.changeDynamics(body, -1, lateralFriction=BLOCK_FRICTION, restitution=BLOCK_RESTITUTION, rollingFriction=0.004, spinningFriction=0.004, physicsClientId=client)
    return body


## Run Simulation and Record Videos

Blocks are inserted one at a time, creating a stacking sequence without using artificial constraints.

In [ ]:
client = connect_pybullet()
table_id = create_table(client)
video_paths, writers = make_writers("tabletop_stacking")
blocks = []
records = []
next_release_step = 0
num_blocks = 6

try:
    for step in range(N_STEPS + 1):
        t = step * TIME_STEP
        if len(blocks) < num_blocks and step >= next_release_step:
            blocks.append(create_block(client, len(blocks)))
            next_release_step += int(RELEASE_INTERVAL_SEC * SIM_HZ)

        row = [t, len(blocks)]
        top_z = TABLE_TOP_Z
        lateral_spread = 0.0
        for body in blocks:
            pos, orn = p.getBasePositionAndOrientation(body, physicsClientId=client)
            lin_vel, ang_vel = p.getBaseVelocity(body, physicsClientId=client)
            top_z = max(top_z, pos[2] + BLOCK_SIZE[2] / 2)
            lateral_spread = max(lateral_spread, float(np.linalg.norm(pos[:2])))
            row.extend([pos[0], pos[1], pos[2], lin_vel[0], lin_vel[1], lin_vel[2]])
        records.append(row)

        if step % STEPS_PER_FRAME == 0:
            target = [0.0, 0.0, TABLE_TOP_Z + 0.18]
            for name, writer in writers.items():
                writer.append_data(render_rgb(client, name, target))

        p.stepSimulation(physicsClientId=client)
finally:
    for writer in writers.values():
        writer.close()
    p.disconnect(client)

max_cols = 2 + num_blocks * 6
padded = np.full((len(records), max_cols), np.nan)
for i, row in enumerate(records):
    padded[i, :len(row)] = row
columns = ["time", "active_blocks"]
for i in range(num_blocks):
    columns.extend([f"block_{i}_x", f"block_{i}_y", f"block_{i}_z", f"block_{i}_vx", f"block_{i}_vy", f"block_{i}_vz"])

trajectory_path = OUTPUT_DIR / "tabletop_stacking_trajectory.csv"
np.savetxt(trajectory_path, padded, delimiter=",", header=",".join(columns), comments="")

print("Saved videos:")
for name, path in video_paths.items():
    print(f"  {name:>5}: {path}")
print(f"Saved trajectory: {trajectory_path}")


## Quick Stability Checks

In [ ]:
final = padded[-1]
positions = final[2:].reshape(-1, 6)[:, :3]
positions = positions[~np.isnan(positions[:, 0])]
top_height = np.max(positions[:, 2] + BLOCK_SIZE[2] / 2)
lateral_offsets = np.linalg.norm(positions[:, :2], axis=1)
max_lateral_offset = np.max(lateral_offsets)
ideal_height = TABLE_TOP_Z + len(positions) * BLOCK_SIZE[2]

print(f"Final active blocks: {len(positions)}")
print(f"Ideal stack height:  {ideal_height:.3f} m")
print(f"Final top height:    {top_height:.3f} m")
print(f"Max lateral offset:  {max_lateral_offset:.3f} m")
print("Small lateral offset and near-ideal height indicate a settled stack; large offsets indicate toppling/sliding.")


## Preview Videos

In [ ]:
for name in CAMERA_NAMES:
    print(name)
    display(Video(str(video_paths[name]), embed=True, html_attributes="controls loop"))
